# Lesson 1 — 예측 문제와 시간 피처의 기초

**예상 시간:** 개념 35분 + 실습 60분  
**오늘의 새 산출물:** 안전한 원본 feature와 시간 파생 feature의 공통 validation 비교

이 notebook은 완성 답안이 아닙니다. toy example로 패턴을 익힌 뒤 실제 데이터 조건에 맞게 `Sparta/competitions/bike-sharing-demand/answers/code/lesson1.ipynb`에서 직접 구현하세요.

## 1. 오늘의 질문

> 2012년 어느 날짜와 시각의 자전거 대여량을 예측한다면, 그 순간 이미 알 수 있는 정보는 무엇이고 `datetime`에서 어떤 단서를 꺼낼 수 있을까?

머신러닝 모델은 표의 한 행을 하나의 연습문제로 봅니다. 이 데이터에서 한 행은 **특정 날짜의 특정 한 시간**이고, `count`는 그 시간에 발생한 총 대여량입니다. Feature는 모델이 문제를 풀 때 보는 단서이고 target은 맞혀야 할 정답입니다.

## 2. 선수 지식 확인

- pandas로 CSV를 읽고 `shape`, `head`, `dtypes`, 결측치를 확인할 수 있는가?
- 실제값과 예측값을 구분할 수 있는가?
- training data와 validation data의 역할을 구분할 수 있는가?

완벽히 기억나지 않아도 괜찮습니다. 오늘 필요한 부분은 아래 설명과 toy example에서 다시 확인합니다.

## 3. 개념 설명

### 3.1 왜 시간 feature가 필요한가?

`2012-10-16 08:00:00`은 사람에게는 화요일 아침이라는 뜻이지만, 원래 문자열 그대로는 많은 모델이 그 의미를 이해하지 못합니다. `year`, `month`, `day`, `hour`, `weekday`로 나누면 모델은 출근 시간과 주말 같은 반복 규칙을 비교할 수 있습니다.

적용 전에는 모델이 긴 날짜 문자열 하나를 받거나 날짜를 통째로 버립니다. 적용 후에는 연도, 월, 일, 시각, 요일이라는 서로 다른 단서를 받습니다. 단, feature를 만들 수 있다는 사실이 성능 향상을 보장하지는 않습니다. 같은 validation 행에서 RMSLE가 낮아졌을 때만 도움이 됐다고 판단합니다.

### 3.2 `pd.to_datetime`과 `.dt`

- `pd.to_datetime(series)`: 문자열을 datetime 값으로 변환한 새 `Series`를 반환합니다.
- 원본 컬럼을 자동 변경하지 않으므로 결과를 다시 저장해야 합니다.
- `.dt.year`, `.dt.month`, `.dt.day`, `.dt.hour`, `.dt.weekday`: datetime Series에서 구성 요소를 꺼낸 새 Series입니다.
- `.dt.weekday`는 월요일 0부터 일요일 6까지입니다.

### 3.3 Target Leakage

`casual + registered = count`입니다. 두 컬럼은 정답의 구성요소이므로 feature로 넣으면 모델이 예측이 아니라 정답 덧셈을 배우게 됩니다. 공식 test에는 두 컬럼도 없습니다. 따라서 `casual`, `registered`, `count`, 원본 `datetime`은 모델 입력에서 제외합니다.

### 3.4 왜 무작위 분할을 피하는가?

대회의 test는 매월 20일 이후입니다. 무작위 분할은 미래 날짜를 train에 넣고 더 이른 날짜를 validation으로 만들 수 있습니다. 오늘은 2012년 10~12월 각각에서 16~19일을 validation으로 두고, 그 validation 시작 이전 행만 train으로 쓰는 expanding backtest를 사용합니다.

## 4. 손으로 만드는 작은 표

| datetime 원문 | year | month | day | hour | weekday |
|---|---:|---:|---:|---:|---:|
| 2024-03-04 08:00:00 | 2024 | 3 | 4 | 8 | 0 |
| 2024-03-09 18:00:00 | 2024 | 3 | 9 | 18 | 5 |

사람이 손으로 한다면 문자열에서 날짜와 시간을 읽고 각 구성요소를 별도 칸에 적습니다. pandas의 datetime 변환과 `.dt` 접근자는 이 작업을 모든 행에 반복합니다.

## 5. 실행 가능한 toy example

아래 자료는 실제 과제와 다른 가상 도서관 방문 데이터입니다. 반환 type과 원본 변경 여부를 확인하세요.

In [ ]:
from pathlib import Path
import pandas as pd

print('현재 작업 폴더:', Path.cwd())

toy = pd.DataFrame({
    'visit_time': ['2024-03-04 08:00:00', '2024-03-09 18:00:00'],
    'rain': [0, 1],
    'visitors': [12, 7],
})

parsed = pd.to_datetime(toy['visit_time'])
print(type(parsed), parsed.dtype)
print('변환 전 원본 dtype:', toy['visit_time'].dtype)

toy_features = toy.copy()
toy_features['visit_time'] = parsed
toy_features['year'] = toy_features['visit_time'].dt.year
toy_features['month'] = toy_features['visit_time'].dt.month
toy_features['day'] = toy_features['visit_time'].dt.day
toy_features['hour'] = toy_features['visit_time'].dt.hour
toy_features['weekday'] = toy_features['visit_time'].dt.weekday

print(toy_features)
print('원본 컬럼:', toy.columns.tolist())
print('새 표 컬럼:', toy_features.columns.tolist())

`toy.copy()`는 새 DataFrame을 반환하므로 `toy_features`에 저장했습니다. `pd.to_datetime`도 새 Series를 반환합니다. 결과를 저장하지 않으면 원래 문자열 컬럼은 바뀌지 않습니다.

대표적인 오류는 `.dt`를 문자열 Series에 바로 쓰는 것입니다. 먼저 `pd.to_datetime`으로 변환해야 합니다.

### 5.1 첫 모델·RMSLE·expanding backtest toy workflow

`DummyRegressor(strategy='mean')`는 train target 평균만 예측하는 최소 기준입니다. Feature를 사용하는 모델이 이 기준도 넘지 못한다면 복잡한 feature의 가치부터 주장할 수 없습니다.

`RandomForestRegressor`는 여러 decision tree의 예측을 합치는 회귀 모델입니다. 여기서는 최고의 모델을 찾기 위해서가 아니라 feature set만 바꿨을 때 결과가 달라지는지 확인하는 고정 측정 도구로 사용합니다.

- 생성자 주요 인수: `n_estimators`는 tree 수, `random_state`는 재현 가능한 난수 기준입니다.
- `.fit(X, y)`: 모델 내부 상태를 학습하고 fitted model 자신을 반환합니다. 입력 DataFrame을 바꾸지 않습니다.
- `.predict(X)`: 새 `numpy.ndarray` 예측값을 반환하므로 변수에 저장해야 합니다.
- RMSLE는 `log1p(actual)`과 `log1p(prediction)`의 차이를 측정합니다. 음수 예측은 허용되지 않으므로 평가 전에 0 이상으로 제한합니다.

Expanding backtest는 validation 시점이 뒤로 갈수록 그 전에 새로 관측된 actual을 train에 추가합니다. 각 fold에서 반드시 새 모델을 만들고 validation 시작 이전 행만 `fit`합니다. 아래는 실제 자전거 데이터와 다른 가상 식당 방문 데이터입니다.

In [ ]:
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_log_error

restaurant = pd.DataFrame({
    'date': pd.date_range('2024-01-01', periods=12, freq='D'),
    'temperature': [5, 7, 6, 8, 9, 11, 10, 12, 13, 12, 14, 15],
    'weekend': [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1],
    'visitors': [20, 22, 21, 25, 27, 38, 42, 29, 31, 32, 34, 48],
}).sort_values('date').reset_index(drop=True)

folds = [
    ('2024-01-07', '2024-01-08'),
    ('2024-01-11', '2024-01-12'),
]
features = ['temperature', 'weekend']
oof_parts = []

for fold_number, (start, end) in enumerate(folds, start=1):
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)
    train_mask = restaurant['date'] < start
    valid_mask = restaurant['date'].between(start, end)
    train_fold = restaurant.loc[train_mask]
    valid_fold = restaurant.loc[valid_mask]

    baseline = DummyRegressor(strategy='mean')
    model = RandomForestRegressor(n_estimators=30, random_state=42)
    baseline.fit(train_fold[features], train_fold['visitors'])
    model.fit(train_fold[features], train_fold['visitors'])

    baseline_pred = np.clip(baseline.predict(valid_fold[features]), 0, None)
    model_pred = np.clip(model.predict(valid_fold[features]), 0, None)
    fold_result = valid_fold[['date', 'visitors']].copy()
    fold_result['baseline_pred'] = baseline_pred
    fold_result['model_pred'] = model_pred
    fold_result['fold'] = fold_number
    oof_parts.append(fold_result)

oof = pd.concat(oof_parts, ignore_index=True)
for prediction_column in ['baseline_pred', 'model_pred']:
    rmsle = np.sqrt(mean_squared_log_error(oof['visitors'], oof[prediction_column]))
    mae = mean_absolute_error(oof['visitors'], oof[prediction_column])
    print(prediction_column, 'OOF RMSLE=', round(rmsle, 4), 'OOF MAE=', round(mae, 4))

print(oof)
print('predict 반환 type:', type(model_pred), 'shape:', model_pred.shape)

이 toy workflow에서 두 번째 fold의 train은 첫 validation 날짜의 actual을 포함합니다. 이는 시간이 지난 뒤 실제값이 관측됐다고 가정하는 expanding 평가입니다. 반대로 두 번째 validation 이후 값은 train에 들어가지 않습니다.

`pd.concat`은 fold 결과를 연결한 새 DataFrame을 반환하므로 `oof`에 저장했습니다. OOF(Out-of-Fold) 예측은 각 행이 자신보다 앞선 train으로 학습한 모델에서 나온 validation 예측입니다. 실제 Exercise에서는 같은 validation 행에 대해 평균 baseline, Feature Set A, Feature Set B를 비교합니다.

## 6. Data Leakage 점검

- `casual`, `registered`, `count`가 `X`에 들어가지 않았는가?
- validation 시작 이후 행이 해당 fold의 train에 들어가지 않았는가?
- `test.csv`의 행이나 예측 결과로 feature를 선택하지 않았는가?
- 날짜 정렬 전에 행 위치로 split하지 않았는가?

`season`, `holiday`, `workingday`, 날씨 예보가 실제 운영 시점에 알려지는 방식은 현실 시스템에서 별도 확인이 필요합니다. 이번 대회에서는 test 입력으로 제공되므로 사용 가능하다고 가정하되 이 가정을 글에 남깁니다.

## 7. 실제 데이터 Exercise

새 모델링 산출물은 **평균 baseline, Feature Set A와 B의 out-of-fold RMSLE/MAE 비교표**입니다. 데이터 점검은 이를 만들기 위한 필수 단계이지 별도 목표가 아닙니다.

- 입력: `Sparta/competitions/bike-sharing-demand/data/train.csv`
- 목표: `count`
- 고정 모델: `RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)`
- Feature Set A: `season`, `holiday`, `workingday`, `weather`, `temp`, `atemp`, `humidity`, `windspeed`
- Feature Set B: A + `year`, `month`, `day`, `hour`, `weekday`
- 비교 fold: `Sparta/competitions/bike-sharing-demand/PLAN.md`의 3개 fold

위 toy workflow의 패턴을 실제 날짜·컬럼·feature set에 맞게 직접 적용하세요. 실제 Exercise의 완성 코드는 제공되지 않습니다. 막히면 Hint부터 요청하세요.

## 8. 코드 과제

**제출 경로:** `Sparta/competitions/bike-sharing-demand/answers/code/lesson1.ipynb`

- **C1 — 데이터 계약:** 저장소 상대경로로 train을 읽고 shape, 컬럼, 날짜 범위, 결측치 수, 날짜 오름차순 여부, `casual + registered == count` 여부를 출력한다.
- **C2 — 시간 feature:** 원본을 보존한 새 DataFrame에 `year`, `month`, `day`, `hour`, `weekday`를 만들고 첫 3행과 dtype을 출력한다.
- **C3 — Fold 검증:** 각 fold의 train/validation 시작·끝 날짜, 행 수, `train.max() < validation.min()` 여부를 출력한다.
- **C4 — 공통 모델 비교:** 각 fold에서 평균 `DummyRegressor` baseline과 Feature Set A/B의 `RandomForestRegressor`를 학습한다. A/B는 동일한 모델 설정을 사용한다. 예측값은 0 이상으로 제한하고 fold별 RMSLE/MAE를 저장한다.
- **C5 — OOF 비교:** baseline/A/B 각각의 모든 validation 실제값과 예측값을 연결해 전체 out-of-fold RMSLE/MAE 및 비교표를 출력한다.

**코드 최소 통과 기준**

- 처음부터 끝까지 실행되고 절대경로가 없다.
- `casual`, `registered`, `count`, 원본 `datetime`이 `X`에 없다.
- 각 fold 모델이 validation 이전 행만 학습한다.
- baseline/A/B가 동일한 validation 행에서 비교되고 A/B 모델 설정이 같다.
- fold별 점수와 전체 OOF 점수가 모두 출력된다.

## 9. 글 과제

**제출 경로:** `Sparta/competitions/bike-sharing-demand/answers/text/lesson1.txt`

- **T1:** `2012-10-16 08:00:00` 행을 예로 들어 한 행, prediction time, target `count`, 사용할 수 있는 원본 단서를 설명한다. **통과 기준:** feature와 target을 구분하고 예측 시점 이후 정보는 사용할 수 없다고 쓴다.
- **T2:** `casual`, `registered`를 feature에서 제외한 이유를 설명한다. **통과 기준:** 두 값의 합이 `count`라는 관계와 test에 없다는 사실을 모두 언급한다.
- **T3:** baseline과 Feature Set A/B의 OOF RMSLE와 MAE를 수치로 적고 시간 feature 채택 여부를 판단한다. **통과 기준:** 최소 기준을 넘었는지와 A/B 중 낮은 validation 오차를 정확히 판단하고 training 성능을 근거로 사용하지 않는다.
- **T4:** 무작위 분할 대신 expanding fold를 쓴 이유를 설명한다. **통과 기준:** 미래 행이 과거 예측의 train에 들어가는 문제와 대회의 월별 시간 구조를 연결한다.

## 10. 성찰 질문

**제출 불필요:** `hour`가 성능을 크게 개선했다면, 그것은 시간이 수요의 원인이라는 증거일까요, 아니면 예측에 유용한 반복 패턴을 발견했다는 증거일까요?

## 11. 제출 전 자체 점검

- [ ] C1~C5가 모두 실행되는가?
- [ ] T1~T4에 실제 출력 수치와 지정 날짜를 사용했는가?
- [ ] Feature Set A/B 외의 피처나 다른 모델을 몰래 추가하지 않았는가?
- [ ] 관찰한 성능과 경제적 원인 추정을 구분했는가?
- [ ] 답안을 지정 경로에 저장했는가?